In [72]:
file_path = 'enron1/ham/0007.1999-12-14.farmer.ham.txt'

In [73]:
with open(file_path, 'r') as infile:
    ham_sample = infile.read()

In [74]:
print(ham_sample)

Subject: mcmullen gas for 11 / 99
jackie ,
since the inlet to 3 river plant is shut in on 10 / 19 / 99 ( the last day of
flow ) :
at what meter is the mcmullen gas being diverted to ?
at what meter is hpl buying the residue gas ? ( this is the gas from teco ,
vastar , vintage , tejones , and swift )
i still see active deals at meter 3405 in path manager for teco , vastar ,
vintage , tejones , and swift
i also see gas scheduled in pops at meter 3404 and 3405 .
please advice . we need to resolve this as soon as possible so settlement
can send out payments .
thanks


In [75]:
file_path = 'enron1/spam/0058.2003-12-21.GP.spam.txt'

In [76]:
with open(file_path, 'r') as infile:
    spam_sample = infile.read()

In [77]:
print(spam_sample)

Subject: stacey automated system generating 8 k per week parallelogram
people are
getting rich using this system ! now it ' s your
turn !
we ' ve
cracked the code and will show you . . . .
this is the
only system that does everything for you , so you can make
money
. . . . . . . .
because your
success is . . . completely automated !
let me show
you how !
click
here
to opt out click here % random _ text



In [78]:
import glob
import os

In [79]:
emails, labels = [], []

In [80]:
file_path = 'enron1/spam'

In [81]:
for filename in glob.glob(os.path.join(file_path, '*.txt')):
    with open(filename, 'r', encoding = 'ISO-8859-1') as infile:
        emails.append(infile.read())
        labels.append(0)

In [82]:
len(emails)

1500

In [83]:
len(labels)

1500

In [84]:
from nltk.corpus import names
from nltk.stem import WordNetLemmatizer
import nltk

In [85]:
def letters_only(astr):
    return astr.isalpha()

In [86]:
nltk.download('names')
nltk.download('wordnet')
all_names = set(names.words())

[nltk_data] Downloading package names to /home/codespace/nltk_data...
[nltk_data]   Package names is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [87]:
lemmatizer = WordNetLemmatizer()

In [96]:
def clean_text(docs):
    cleaned_docs = []
    for doc in docs:
        cleaned_docs.append(' '.join([lemmatizer.lemmatize(word.lower()) 
                                      for word in doc.split() 
                                      if letters_only(word) 
                                      and word not in all_names]))
    return cleaned_docs

In [97]:
cleaned_emails = clean_text(emails)

In [98]:
cleaned_emails[0]

'fw don t be a fuddy duddy use the sof tware everyone s using oemwhat is it oem stand for original equipment manufacturer it primarily refers to name brand software that come without the box or owner s manual this isn t a problem because of the time one can find the manual to download right online from the manufacturer website why do you care you can purchase oemincluding microsoft microsoft office adobe macromedia corel even title for the macintosh for unbelievably low price afraction of what it costsfrom the original manufacturer you need to see it to bel ieve it you candownload it straight from this siteby going here keep in mind you ll need to burnthe isoto a cd if you don t have a cd burneryou can go hereand have themmail it rightto your doorstep at no extra cost cancel auto update follow this one'

In [99]:
from sklearn.feature_extraction.text import CountVectorizer

In [100]:
cv = CountVectorizer(stop_words = 'english', max_features = 500)

In [101]:
term_docs = cv.fit_transform(cleaned_emails)

In [102]:
print(term_docs.shape)

(1500, 500)


In [103]:
print(len(cleaned_emails))

1500


In [104]:
feature_names = cv.get_feature_names_out()

In [105]:
feature_names[60]

'color'

In [106]:
feature_mapping = cv.vocabulary_

In [107]:
def get_label_index(labels):
    from collections import defaultdict
    label_index = defaultdict(list)
    for index, label in enumerate(labels):
        label_index[label].append(index)
    return label_index
label_index = get_label_index(labels)

In [108]:
def get_prior(label_index):
    """Compute prior based on training samples
    Args:
        label_index (grouped sample infices by class)
    Returns:
        dictionary, with class label as key, corresponding
        prior as the value
    """

    prior = {label: len(index) for label, index in label_index.items()}
    total_count = sum(prior.values())
    for label in prior:
        prior[label] /= float(total_count)
    return prior

In [109]:
prior = get_prior(label_index)

In [110]:
prior

{0: 1.0}

In [111]:
import numpy as np

In [112]:
def get_likelihood(term_docs, label_index, smoothing = 0):
    """Compute likelihood based on training samples
    Args:
        term_docs (sparse matrix)
        label_index (grouped sample infices by class)
        smoothing (integer, additive Laplace smoothing parameter)
    Returns:
        dictionary, with class label as key, corresponding
        conditional probability P(feature|class) vector as the value
    """
    likelihood = {}
    for label, index in label_index.items():
        likelihood[label] = term_docs[index, :].sum(axis = 0) + smoothing
        likelihood[label] = np.asarray(likelihood[label])[0]
        total_count = likelihood[label].sum()
        likelihood[label] = likelihood[label] / float(total_count)
    return likelihood

In [113]:
smoothing = 1

In [114]:
likelihood = get_likelihood(term_docs, label_index, smoothing)

In [115]:
len(likelihood[0])

500

In [116]:
likelihood[0][:5]

array([0.00085199, 0.0011092 , 0.00356873, 0.00083592, 0.00329545])

In [117]:
likelihood[1][:5]

KeyError: 1